In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!pip install torch matplotlib scipy -q

# Experiment Overview

Experiment 14 — ZK on Partially Communicating Systems

Motivation:
  Exp 2 tested the ZK metric on two extreme cases:
  - Fully communicating game (ZK >> 0)
  - Severed channel baseline (ZK ≈ 0)
  
  But the metric is most useful in the AMBIGUOUS MIDDLE — partially
  communicating systems. This experiment:
  
  1. Trains games with progressively bottlenecked channels:
     - Message vocabulary size: 2, 4, 8, 16, 32
     - Partial information: sender sees only k of N features
  2. Plots ZK score against task accuracy
  3. Tests whether ZK tracks communication quality CONTINUOUSLY
     or just detects its presence/absence

  If ZK is continuous: it becomes a TRAINING SIGNAL, not just a diagnostic.
  If binary: it only tells you yes/no communication, which is less useful.

In [ ]:
import os, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
device = 'cpu'
os.makedirs('results', exist_ok=True)

# ── Hyperparams ───────────────────────────────────────────────────────────────
N_FEATURES    = 16
N_DISTRACTORS = 4
N_TRAIN       = 8000
N_VAL         = 2000
QUERY_DIM     = 16
HIDDEN        = 64
N_EPOCHS      = 60
BATCH_SIZE    = 128
LR            = 1e-3
GS_TEMP       = 0.5
N_SCRAMBLES   = 20

# ── Data ──────────────────────────────────────────────────────────────────────
def make_dataset(n_samples, seed=42):
    rng = np.random.default_rng(seed)
    objects = rng.integers(0, 2,
        (n_samples, N_DISTRACTORS+1, N_FEATURES)).astype(np.float32)
    sender_input = torch.tensor(objects[:, 0, :])
    labels       = torch.zeros(n_samples, dtype=torch.long)
    recv_input   = torch.tensor(objects)
    return TensorDataset(sender_input, labels, recv_input)

train_ds = make_dataset(N_TRAIN, seed=42)
val_ds   = make_dataset(N_VAL,   seed=99)

# ── Agents (parameterized vocab size) ─────────────────────────────────────────
class Sender(nn.Module):
    def __init__(self, vocab_size=16, input_dim=N_FEATURES):
        super().__init__()
        self.turn1 = nn.Sequential(
            nn.Linear(input_dim, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, vocab_size)
        )
        self.turn2 = nn.Sequential(
            nn.Linear(input_dim + QUERY_DIM, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, vocab_size)
        )
        self.vocab_size = vocab_size

    def forward_turn1(self, obj):
        return self.turn1(obj)

    def forward_turn2(self, obj, query):
        return self.turn2(torch.cat([obj, query], dim=-1))


class Receiver(nn.Module):
    def __init__(self, vocab_size=16):
        super().__init__()
        self.msg_proj  = nn.Linear(vocab_size, HIDDEN)
        self.obj_proj  = nn.Linear(N_FEATURES, HIDDEN)
        self.query_mlp = nn.Sequential(
            nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, QUERY_DIM), nn.Tanh()
        )
        self.vocab_size = vocab_size

    def _attend(self, msg_h, recv_input):
        obj_h  = self.obj_proj(recv_input)
        scores = torch.bmm(obj_h, msg_h.unsqueeze(-1)).squeeze(-1)
        return scores

    def forward_turn1(self, msg, recv_input):
        msg_h  = self.msg_proj(msg)
        query  = self.query_mlp(msg_h)
        scores = self._attend(msg_h, recv_input)
        return query, scores

    def forward_turn2(self, msg1, msg2, recv_input):
        h1 = self.msg_proj(msg1)
        h2 = self.msg_proj(msg2)
        return self._attend((h1 + h2) / 2, recv_input)


class MultiTurnGame(nn.Module):
    def __init__(self, sender, receiver, severed=False, partial_features=None):
        super().__init__()
        self.sender   = sender
        self.receiver = receiver
        self.severed  = severed
        self.partial_features = partial_features  # if set, sender only sees k features
        self.vocab_size = sender.vocab_size

    def forward(self, si, labels, ri, tau=GS_TEMP):
        B = si.size(0)
        
        # Optionally mask sender input (partial information)
        if self.partial_features is not None:
            mask = torch.zeros_like(si)
            mask[:, :self.partial_features] = 1.0
            si_masked = si * mask
        else:
            si_masked = si

        # Turn 1
        logits1 = self.sender.forward_turn1(si_masked)
        if self.severed:
            msg1 = torch.zeros(B, self.vocab_size)
        elif self.training:
            msg1 = F.gumbel_softmax(logits1, tau=tau, hard=True)
        else:
            msg1 = F.one_hot(logits1.argmax(-1), self.vocab_size).float()

        query, scores1 = self.receiver.forward_turn1(msg1, ri)

        # Turn 2
        logits2 = self.sender.forward_turn2(si_masked, query)
        if self.severed:
            msg2 = torch.zeros(B, self.vocab_size)
        elif self.training:
            msg2 = F.gumbel_softmax(logits2, tau=tau, hard=True)
        else:
            msg2 = F.one_hot(logits2.argmax(-1), self.vocab_size).float()

        scores2 = self.receiver.forward_turn2(msg1, msg2, ri)

        loss = F.cross_entropy(scores2, labels)
        acc  = (scores2.argmax(-1) == labels).float().mean()
        return loss, acc, logits1, logits2, msg1, msg2, query, scores1, scores2


# ── Training ──────────────────────────────────────────────────────────────────
def train_game(vocab_size=16, severed=False, partial_features=None, label=''):
    input_dim = N_FEATURES
    sender   = Sender(vocab_size=vocab_size, input_dim=input_dim)
    receiver = Receiver(vocab_size=vocab_size)
    game     = MultiTurnGame(sender, receiver, severed=severed,
                             partial_features=partial_features)
    opt      = torch.optim.Adam(game.parameters(), lr=LR)

    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    for epoch in range(N_EPOCHS):
        game.train()
        for si, lb, ri in loader:
            opt.zero_grad()
            loss, *_ = game(si, lb, ri)
            loss.backward()
            opt.step()

        if (epoch+1) % 20 == 0:
            game.eval()
            accs = []
            with torch.no_grad():
                for si, lb, ri in DataLoader(val_ds, batch_size=256):
                    _, acc, *_ = game(si, lb, ri)
                    accs.append(acc.item())
            print(json.dumps({'label': label, 'epoch': epoch+1,
                              'val_acc': round(float(np.mean(accs)), 4)}))
    return game


# ── ZK Metric ─────────────────────────────────────────────────────────────────
def js_divergence(p_logits, q_logits):
    p = F.softmax(p_logits, dim=-1)
    q = F.softmax(q_logits, dim=-1)
    m = 0.5 * (p + q)
    js = 0.5 * (p * (p / m.clamp(1e-9)).log()).sum(-1) + \
         0.5 * (q * (q / m.clamp(1e-9)).log()).sum(-1)
    return js.mean().item()


def compute_zk_metric(game, n_scrambles=N_SCRAMBLES):
    game.eval()
    loader = DataLoader(val_ds, batch_size=256, shuffle=False)
    vocab_size = game.vocab_size

    cic1_all, rc_all, cic2_all = [], [], []

    with torch.no_grad():
        for si, lb, ri in loader:
            B = si.size(0)
            
            # Mask sender input if partial
            if game.partial_features is not None:
                mask = torch.zeros_like(si)
                mask[:, :game.partial_features] = 1.0
                si_m = si * mask
            else:
                si_m = si

            logits1_real = game.sender.forward_turn1(si_m)
            msg1_real    = F.one_hot(logits1_real.argmax(-1), vocab_size).float()
            query_real, _ = game.receiver.forward_turn1(msg1_real, ri)
            logits2_real  = game.sender.forward_turn2(si_m, query_real)
            msg2_real     = F.one_hot(logits2_real.argmax(-1), vocab_size).float()
            scores2_real  = game.receiver.forward_turn2(msg1_real, msg2_real, ri)

            for _ in range(n_scrambles):
                perm = torch.randperm(B)

                msg1_fake     = msg1_real[perm]
                query_fake, _ = game.receiver.forward_turn1(msg1_fake, ri)
                delta_query   = (query_real - query_fake).pow(2).sum(-1).sqrt()
                cic1_all.append(delta_query.mean().item())

                logits2_fake = game.sender.forward_turn2(si_m, query_fake)
                rc_all.append(js_divergence(logits2_real, logits2_fake))

                msg2_fake    = msg2_real[perm]
                scores2_fake = game.receiver.forward_turn2(msg1_real, msg2_fake, ri)
                delta_scores = (scores2_real - scores2_fake).pow(2).sum(-1).sqrt()
                cic2_all.append(delta_scores.mean().item())

    cic1 = float(np.mean(cic1_all))
    rc   = float(np.mean(rc_all))
    cic2 = float(np.mean(cic2_all))
    zk   = ((cic1 + cic2) / 2) * rc

    return {
        'cic_turn1': round(cic1, 5),
        'rc_coeff':  round(rc,   5),
        'cic_turn2': round(cic2, 5),
        'zk_score':  round(zk,   5),
    }


def eval_accuracy(game):
    game.eval()
    accs = []
    with torch.no_grad():
        for si, lb, ri in DataLoader(val_ds, batch_size=256):
            _, acc, *_ = game(si, lb, ri)
            accs.append(acc.item())
    return float(np.mean(accs))

## SWEEP 1: Vocabulary bottleneck (2, 4, 8, 16, 32)

In [ ]:
print('═'*60)
print('SWEEP 1: Vocabulary size bottleneck')
print('═'*60)

VOCAB_SIZES = [2, 4, 8, 16, 32]
vocab_results = []

for vs in VOCAB_SIZES:
    print(f'\n--- Vocab size = {vs} ---')
    g = train_game(vocab_size=vs, label=f'vocab_{vs}')
    acc = eval_accuracy(g)
    zk = compute_zk_metric(g)
    vocab_results.append({
        'vocab_size': vs, 'accuracy': round(acc, 4), **zk
    })
    print(f'  Acc={acc:.4f}  ZK={zk["zk_score"]:.5f}  CIC1={zk["cic_turn1"]:.4f}  RC={zk["rc_coeff"]:.5f}  CIC2={zk["cic_turn2"]:.4f}')

# Also train severed baseline
print('\n--- Severed baseline ---')
sev_game = train_game(vocab_size=16, severed=True, label='severed')
sev_acc = eval_accuracy(sev_game)
sev_zk = compute_zk_metric(sev_game)
print(f'  Acc={sev_acc:.4f}  ZK={sev_zk["zk_score"]:.5f}')

## SWEEP 2: Partial information (sender sees k of 16 features)

In [ ]:
print('\n' + '═'*60)
print('SWEEP 2: Partial sender information')
print('═'*60)

PARTIAL_FEATURES = [2, 4, 8, 12, 16]
partial_results = []

for pf in PARTIAL_FEATURES:
    print(f'\n--- Sender sees {pf}/{N_FEATURES} features ---')
    g = train_game(vocab_size=16, partial_features=pf, label=f'partial_{pf}')
    acc = eval_accuracy(g)
    zk = compute_zk_metric(g)
    partial_results.append({
        'partial_features': pf, 'accuracy': round(acc, 4), **zk
    })
    print(f'  Acc={acc:.4f}  ZK={zk["zk_score"]:.5f}  CIC1={zk["cic_turn1"]:.4f}  RC={zk["rc_coeff"]:.5f}  CIC2={zk["cic_turn2"]:.4f}')

## Analysis: Is ZK continuous?

In [ ]:
print('\n' + '═'*60)
print('ANALYSIS: Is ZK a continuous metric?')
print('═'*60)

# Compute Spearman correlation between ZK and accuracy
from scipy.stats import spearmanr

# Sweep 1
accs_v = [r['accuracy'] for r in vocab_results]
zks_v  = [r['zk_score'] for r in vocab_results]
rho_v, pval_v = spearmanr(accs_v, zks_v)
print(f'\nVocab sweep: Spearman ρ(accuracy, ZK) = {rho_v:.4f}  (p={pval_v:.4f})')

# Sweep 2
accs_p = [r['accuracy'] for r in partial_results]
zks_p  = [r['zk_score'] for r in partial_results]
rho_p, pval_p = spearmanr(accs_p, zks_p)
print(f'Partial sweep: Spearman ρ(accuracy, ZK) = {rho_p:.4f}  (p={pval_p:.4f})')

# Combined
all_accs = accs_v + accs_p + [sev_acc]
all_zks  = zks_v + zks_p + [sev_zk['zk_score']]
rho_all, pval_all = spearmanr(all_accs, all_zks)
print(f'Combined: Spearman ρ(accuracy, ZK) = {rho_all:.4f}  (p={pval_all:.4f})')

if rho_all > 0.7:
    print('\nVERDICT: ZK IS CONTINUOUS — it tracks communication quality, not just presence.')
    print('→ ZK can be used as a training signal (auxiliary loss for communication quality)')
elif rho_all > 0.4:
    print('\nVERDICT: PARTIALLY CONTINUOUS — ZK correlates with quality but with noise.')
    print('→ ZK is a useful diagnostic but not reliable enough as a training signal')
else:
    print('\nVERDICT: ZK IS BINARY — it detects communication presence but not quality.')
    print('→ ZK remains a diagnostic tool, not a gradient-compatible training signal')

## Visualisation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: Vocabulary sweep
ax = axes[0, 0]
ax.plot([r['vocab_size'] for r in vocab_results],
        [r['accuracy'] for r in vocab_results],
        'o-', color='#1976D2', linewidth=2, markersize=8)
ax.axhline(sev_acc, color='gray', linestyle='--', alpha=0.5, label='severed')
ax.set_xlabel('Vocabulary size')
ax.set_ylabel('Task accuracy')
ax.set_title('Vocab sweep: Accuracy')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot([r['vocab_size'] for r in vocab_results],
        [r['zk_score'] for r in vocab_results],
        'o-', color='#E24B4A', linewidth=2, markersize=8)
ax.axhline(sev_zk['zk_score'], color='gray', linestyle='--', alpha=0.5, label='severed')
ax.set_xlabel('Vocabulary size')
ax.set_ylabel('ZK score')
ax.set_title('Vocab sweep: ZK metric')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 2]
ax.scatter([r['accuracy'] for r in vocab_results],
           [r['zk_score'] for r in vocab_results],
           s=100, c='#1976D2', zorder=3, label='vocab sweep')
ax.scatter([sev_acc], [sev_zk['zk_score']], s=100, c='gray',
           marker='x', zorder=3, label='severed')
for r in vocab_results:
    ax.annotate(f'v={r["vocab_size"]}', (r['accuracy']+0.005, r['zk_score']+0.01), fontsize=8)
ax.set_xlabel('Task accuracy')
ax.set_ylabel('ZK score')
ax.set_title(f'Vocab: ZK vs Accuracy\n(ρ={rho_v:.3f})')
ax.legend(); ax.grid(True, alpha=0.3)

# Row 2: Partial information sweep
ax = axes[1, 0]
ax.plot([r['partial_features'] for r in partial_results],
        [r['accuracy'] for r in partial_results],
        'o-', color='#1D9E75', linewidth=2, markersize=8)
ax.axhline(sev_acc, color='gray', linestyle='--', alpha=0.5, label='severed')
ax.set_xlabel('Features visible to sender (of 16)')
ax.set_ylabel('Task accuracy')
ax.set_title('Partial info sweep: Accuracy')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot([r['partial_features'] for r in partial_results],
        [r['zk_score'] for r in partial_results],
        'o-', color='#EF9F27', linewidth=2, markersize=8)
ax.axhline(sev_zk['zk_score'], color='gray', linestyle='--', alpha=0.5, label='severed')
ax.set_xlabel('Features visible to sender')
ax.set_ylabel('ZK score')
ax.set_title('Partial info sweep: ZK metric')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1, 2]
# Combined scatter
ax.scatter(accs_v, zks_v, s=100, c='#1976D2', label='vocab sweep', zorder=3)
ax.scatter(accs_p, zks_p, s=100, c='#1D9E75', marker='s', label='partial info', zorder=3)
ax.scatter([sev_acc], [sev_zk['zk_score']], s=120, c='gray', marker='x',
           linewidths=3, label='severed', zorder=3)
# Add regression line
z = np.polyfit(all_accs, all_zks, 1)
x_line = np.linspace(min(all_accs)*0.9, max(all_accs)*1.1, 100)
ax.plot(x_line, np.polyval(z, x_line), '--', color='gray', alpha=0.5)
ax.set_xlabel('Task accuracy')
ax.set_ylabel('ZK score')
ax.set_title(f'Combined: ZK vs Accuracy\n(ρ={rho_all:.3f}, p={pval_all:.4f})')
ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('Experiment 14 — ZK on Partially Communicating Systems\n'
             'Does the ZK metric track communication quality continuously?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp14_zk_partial_communication.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Component breakdown ───────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5))

# ZK components across vocab sweep
components = ['cic_turn1', 'rc_coeff', 'cic_turn2']
comp_colors = ['#378ADD', '#1D9E75', '#EF9F27']
comp_labels = ['CIC1 (query shift)', 'RC (sender responds)', 'CIC2 (score shift)']

for ci, (comp, color, label) in enumerate(zip(components, comp_colors, comp_labels)):
    axes2[0].plot([r['vocab_size'] for r in vocab_results],
                  [r[comp] for r in vocab_results],
                  'o-', color=color, linewidth=2, markersize=6, label=label)
axes2[0].set_xlabel('Vocabulary size')
axes2[0].set_ylabel('Component value')
axes2[0].set_title('Vocab sweep: ZK components')
axes2[0].legend(fontsize=8); axes2[0].grid(True, alpha=0.3)

for ci, (comp, color, label) in enumerate(zip(components, comp_colors, comp_labels)):
    axes2[1].plot([r['partial_features'] for r in partial_results],
                  [r[comp] for r in partial_results],
                  'o-', color=color, linewidth=2, markersize=6, label=label)
axes2[1].set_xlabel('Features visible to sender')
axes2[1].set_ylabel('Component value')
axes2[1].set_title('Partial info sweep: ZK components')
axes2[1].legend(fontsize=8); axes2[1].grid(True, alpha=0.3)

# Which component is the bottleneck?
axes2[2].bar(['CIC1', 'RC', 'CIC2'],
             [np.std([r['cic_turn1'] for r in vocab_results + partial_results]),
              np.std([r['rc_coeff'] for r in vocab_results + partial_results]),
              np.std([r['cic_turn2'] for r in vocab_results + partial_results])],
             color=comp_colors)
axes2[2].set_ylabel('Std across conditions')
axes2[2].set_title('Component variability\n(most variable = most diagnostic)')
axes2[2].grid(axis='y', alpha=0.3)

plt.suptitle('ZK Metric Component Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp14_zk_components.png', dpi=150, bbox_inches='tight')
plt.show()

## Final summary

In [ ]:
print('\n' + '═'*60)
print('EXPERIMENT 14 — SUMMARY')
print('═'*60)
print('\nVocab sweep results:')
for r in vocab_results:
    print(f'  vocab={r["vocab_size"]:>3}  acc={r["accuracy"]:.3f}  ZK={r["zk_score"]:.5f}')
print(f'  severed      acc={sev_acc:.3f}  ZK={sev_zk["zk_score"]:.5f}')

print('\nPartial information sweep results:')
for r in partial_results:
    print(f'  features={r["partial_features"]:>2}/16  acc={r["accuracy"]:.3f}  ZK={r["zk_score"]:.5f}')

print(f'\nCorrelation analysis:')
print(f'  Vocab sweep:    ρ = {rho_v:.4f}')
print(f'  Partial sweep:  ρ = {rho_p:.4f}')
print(f'  Combined:       ρ = {rho_all:.4f}')

print('\nImplications:')
print('  If ρ > 0.7: ZK is a continuous metric → can be used as training signal')
print('  If ρ < 0.4: ZK is binary → diagnostic only')
print('  The component analysis reveals WHICH link in the causal chain')
print('  (M1→query→M2→decision) is the bottleneck under each degradation mode.')